# South Liwa - Relative Soil Salinity Screening (Sentinel-2)
**Arab Youth Hackathon Challenge 813 - Agriculture / Precision Agriculture**

This notebook is the Python / Colab version of the Google Earth Engine analysis.
It computes **relative** spectral salinity-risk classes, NDVI / NDMI time series,
a reference-farm comparison, a hotspot check, and terrain (dune) statistics.

> **Important:** the indices are *relative*. They do **not** give salinity in % or dS/m.
> Field EC measurements are needed for calibration.

**Before running:** you need an Earth Engine account and a Google Cloud project ID.

In [ ]:
!pip install -q geemap

In [ ]:
import math
import ee
import geemap
import pandas as pd
import matplotlib.pyplot as plt

# Needed for interactive maps in Colab
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

PROJECT = 'YOUR_GOOGLE_CLOUD_PROJECT_ID'   # EDIT: your Earth Engine / Cloud project ID

ee.Authenticate()
ee.Initialize(project=PROJECT)

## 1. Study area, hotspot and parameters

In [ ]:
study_area = ee.Geometry.Rectangle([
    53.8904285443, 23.0742614827,
    53.9404285443, 23.1242614827
])
hotspot1 = ee.Geometry.Point([53.91542854431856, 23.099261482727382])
hs = hotspot1.buffer(30)          # 30 m buffer around the candidate point

OPT_BANDS = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12']
PAL = ['#1a9850', '#91cf60', '#fee08b', '#fc8d59', '#d73027']

print('Study area km2:', round(study_area.area(1).divide(1e6).getInfo(), 2))

## 2. Pre-processing and analysis function

In [ ]:
def prep(img):
    """Cloud mask (SCL) + scale reflectance + bare-soil flag."""
    scl = img.select('SCL')
    clear = (scl.neq(3).And(scl.neq(8)).And(scl.neq(9))
             .And(scl.neq(10)).And(scl.neq(11)))
    optical = img.select(OPT_BANDS).divide(10000)
    bare = scl.eq(5).rename('bare')          # SCL class 5 = bare soil
    return (optical.addBands(bare)
            .updateMask(clear)
            .copyProperties(img, ['system:time_start']))


def analyze(start_date, end_date):
    col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
           .filterBounds(study_area)
           .filterDate(start_date, end_date)
           .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
           .map(prep))

    comp = col.median().clip(study_area)
    bare_frac = col.select('bare').mean().clip(study_area)

    G, R, N = comp.select('B3'), comp.select('B4'), comp.select('B8')

    ndvi = comp.normalizedDifference(['B8', 'B4']).rename('NDVI')
    ndmi = comp.normalizedDifference(['B8', 'B11']).rename('NDMI')
    ndwi = comp.normalizedDifference(['B3', 'B8']).rename('NDWI')

    # Salinity indices (verify formulas against the original papers)
    si = G.multiply(R).sqrt().rename('SI')                      # Douaoui et al. 2006
    ndsi = R.subtract(N).divide(R.add(N)).rename('NDSI')        # Khan et al. 2005

    # Keep mostly-bare, non-vegetated, dry, non-water pixels
    valid = (bare_frac.gt(0.5)
             .And(ndvi.lt(0.15))
             .And(ndmi.lt(0))
             .And(ndwi.lt(0))
             .rename('valid'))

    si_m = si.updateMask(valid)
    ndsi_m = ndsi.updateMask(valid)

    p_si = si_m.reduceRegion(
        reducer=ee.Reducer.percentile([20, 40, 60, 80]),
        geometry=study_area, scale=10, maxPixels=1e9, bestEffort=True)
    p_nd = ndsi_m.reduceRegion(
        reducer=ee.Reducer.percentile([20, 80]),
        geometry=study_area, scale=10, maxPixels=1e9, bestEffort=True)

    p20 = ee.Number(p_si.get('SI_p20'))
    p40 = ee.Number(p_si.get('SI_p40'))
    p60 = ee.Number(p_si.get('SI_p60'))
    p80 = ee.Number(p_si.get('SI_p80'))
    n80 = ee.Number(p_nd.get('NDSI_p80'))

    # 5 relative classes: 1 = lowest, 5 = highest spectral salinity signal
    cls = (ee.Image(1)
           .where(si_m.gte(p20), 2)
           .where(si_m.gte(p40), 3)
           .where(si_m.gte(p60), 4)
           .where(si_m.gte(p80), 5)
           .updateMask(si_m.mask())
           .rename('class'))

    # Candidate = top 20% in BOTH indices
    cand = si_m.gte(p80).And(ndsi_m.gte(n80)).rename('candidate')

    return {'count': col.size(), 'si': si_m, 'ndsi': ndsi_m, 'cls': cls,
            'cand': cand, 'valid': valid, 'ndvi': ndvi, 'ndmi': ndmi}

## 3. Two periods to test persistence

In [ ]:
A = analyze('2026-01-01', '2026-04-30')
B = analyze('2026-05-01', '2026-09-25')

# Persistent candidates: high in both periods
stable = A['cand'].And(B['cand']).rename('stable')

print('Images used (A, B):', A['count'].getInfo(), B['count'].getInfo())

valid_share = (A['valid'].unmask(0)
               .reduceRegion(reducer=ee.Reducer.mean(), geometry=study_area,
                             scale=10, maxPixels=1e9).get('valid').getInfo())
print('Valid pixel share (A):', valid_share)

ha_img = ee.Image.pixelArea().divide(10000)
persist_ha = (ha_img.updateMask(stable.unmask(0))
              .reduceRegion(reducer=ee.Reducer.sum(), geometry=study_area,
                            scale=10, maxPixels=1e9).get('area').getInfo())
print('Persistent candidate area (ha):', persist_ha)

## 4. Map

In [ ]:
Map = geemap.Map(center=[23.099, 53.915], zoom=13)
Map.add_basemap('HYBRID')
Map.addLayer(ee.FeatureCollection(study_area).style(color='red', fillColor='00000000'),
             {}, 'Study area')
Map.addLayer(A['cls'], {'min': 1, 'max': 5, 'palette': PAL}, 'Class - period A', False)
Map.addLayer(B['cls'], {'min': 1, 'max': 5, 'palette': PAL}, 'Class - period B')
Map.addLayer(stable.selfMask(), {'palette': ['black']}, 'Persistent candidates')
Map.addLayer(hotspot1, {'color': 'blue'}, 'Candidate hotspot')
Map

## 5. Monthly time series (NDVI, NDMI, SI)
Uses cloud-masked images only (no bare-soil / vegetation masking), so vegetation and moisture changes stay visible.

In [ ]:
s2all = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
         .filterBounds(study_area)
         .filterDate('2026-01-01', '2026-10-01')
         .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
         .map(prep))

ts_start = ee.Date('2026-01-01')
N_MONTHS = 9      # Jan - Sep 2026


def month_image(m):
    s = ts_start.advance(m, 'month')
    e = s.advance(1, 'month')
    sub = s2all.filterDate(s, e).select(OPT_BANDS)
    # months without images -> fully masked image (shows as a gap)
    comp = ee.Image(ee.Algorithms.If(
        sub.size().gt(0),
        sub.median(),
        ee.Image.constant([0, 0, 0, 0, 0, 0]).rename(OPT_BANDS).updateMask(0)))
    ndvi = comp.normalizedDifference(['B8', 'B4']).rename('NDVI')
    ndmi = comp.normalizedDifference(['B8', 'B11']).rename('NDMI')
    si_t = comp.select('B3').multiply(comp.select('B4')).sqrt().rename('SI')
    return (ndvi.addBands(ndmi).addBands(si_t)
            .set('system:time_start', s.millis())
            .set('n_images', sub.size()))


monthly = ee.ImageCollection.fromImages(ee.List.sequence(0, N_MONTHS - 1).map(month_image))
print('Images per month:', monthly.aggregate_array('n_images').getInfo())


def series_df(col, region, scale, mask=None):
    """Monthly mean of every band over a region -> pandas DataFrame."""
    def f(img):
        im = img if mask is None else img.updateMask(mask)
        d = im.reduceRegion(reducer=ee.Reducer.mean(), geometry=region,
                            scale=scale, maxPixels=1e10, bestEffort=True)
        return ee.Feature(None, d).set(
            'month', ee.Date(img.get('system:time_start')).get('month'))
    fc = ee.FeatureCollection(col.map(f))
    rows = [x['properties'] for x in fc.getInfo()['features']]
    return pd.DataFrame(rows).sort_values('month').reset_index(drop=True)


df_study = series_df(monthly, study_area, 20)
df_hot = series_df(monthly, hs, 10)
df_study

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(df_study['month'], df_study['NDVI'], 'o-', label='NDVI')
axes[0].plot(df_study['month'], df_study['NDMI'], 'o-', label='NDMI')
axes[0].set_title('Study area - monthly NDVI & NDMI (2026)')

axes[1].plot(df_hot['month'], df_hot['NDVI'], 'o-', label='NDVI')
axes[1].plot(df_hot['month'], df_hot['NDMI'], 'o-', label='NDMI')
axes[1].plot(df_hot['month'], df_hot['SI'], 'o-', label='SI')
axes[1].set_title('Candidate hotspot - NDVI, NDMI & SI (2026)')

axes[2].plot(df_study['month'], df_study['SI'], 'o-', color='#d73027', label='SI')
axes[2].set_title('Study area - monthly SI')

for ax in axes:
    ax.set_xlabel('Month (2026)')
    ax.axhline(0, color='black', lw=0.6)
    ax.grid(alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()

## 6. Reference farms
Pixels with annual NDVI > 0.3 within 30 km of the hotspot are used as reference (likely irrigated vegetation).
**Check them on high-resolution imagery before relying on them.**

In [ ]:
annual_ndvi = (s2all.select(OPT_BANDS).median()
               .normalizedDifference(['B8', 'B4']).rename('NDVI'))
ref_mask = annual_ndvi.gt(0.3).selfMask()
search_area = hotspot1.buffer(30000)

n_ref = (ref_mask.reduceRegion(reducer=ee.Reducer.count(), geometry=search_area,
                               scale=30, maxPixels=1e10, bestEffort=True)
         .get('NDVI').getInfo())
print('Reference pixels (30 m scale):', n_ref)

df_ref = series_df(monthly, search_area, 30, mask=ref_mask)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, band in zip(axes, ['NDVI', 'NDMI']):
    ax.plot(df_hot['month'], df_hot[band], 'o-', label=band + ' hotspot')
    ax.plot(df_study['month'], df_study[band], 'o-', label=band + ' study area')
    ax.plot(df_ref['month'], df_ref[band], 'o-', label=band + ' reference farms')
    ax.set_title(band + ' - hotspot vs study area vs reference farms (2026)')
    ax.set_xlabel('Month (2026)')
    ax.axhline(0, color='black', lw=0.6)
    ax.grid(alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()

## 7. Hotspot check

In [ ]:
refl = (s2all.select(OPT_BANDS).median()
        .reduceRegion(reducer=ee.Reducer.mean(), geometry=hs, scale=10).getInfo())
print('Hotspot mean reflectance:', refl)


def hotspot_rank(res):
    """Share of valid study-area pixels with SI lower than the hotspot.
    ~0.8+ -> top 20 percent;  ~0.5 -> ordinary."""
    hs_si = ee.Number(res['si'].reduceRegion(
        reducer=ee.Reducer.mean(), geometry=hs, scale=10).get('SI'))
    return (res['si'].lt(hs_si)
            .reduceRegion(reducer=ee.Reducer.mean(), geometry=study_area,
                          scale=10, maxPixels=1e9).get('SI').getInfo())


print('Hotspot SI rank - period A:', hotspot_rank(A))
print('Hotspot SI rank - period B:', hotspot_rank(B))

## 8. Project plot and terrain (dunes)
Edit `plot_center` and `plot_area_ha` to move or resize the plot. SRTM is 30 m, so terrain values are indicative.

In [ ]:
plot_center = ee.Geometry.Point([53.9154, 23.0990])   # EDIT: lon, lat
plot_area_ha = 5                                      # EDIT

plot_side = math.sqrt(plot_area_ha * 10000)           # metres
plot = plot_center.buffer(plot_side / 2).bounds()

print('Plot area (ha):', plot.area(1).divide(10000).getInfo())
print('Plot inside study area?', study_area.contains(plot, 1).getInfo())

dem = ee.Image('USGS/SRTMGL1_003').select('elevation')
slope = ee.Terrain.slope(dem)

print('Plot elevation min/max (m):', dem.reduceRegion(
    reducer=ee.Reducer.minMax(), geometry=plot, scale=30, maxPixels=1e9).getInfo())
print('Plot slope mean/max (deg):', slope.reduceRegion(
    reducer=ee.Reducer.mean().combine(ee.Reducer.max(), sharedInputs=True),
    geometry=plot, scale=30, maxPixels=1e9).getInfo())

annual = s2all.select(OPT_BANDS).median()
plot_idx = ee.Image.cat([
    annual.normalizedDifference(['B8', 'B4']).rename('NDVI'),
    annual.normalizedDifference(['B8', 'B11']).rename('NDMI'),
    annual.select('B3').multiply(annual.select('B4')).sqrt().rename('SI')])
print('Plot mean NDVI / NDMI / SI (2026):', plot_idx.reduceRegion(
    reducer=ee.Reducer.mean(), geometry=plot, scale=10, maxPixels=1e9).getInfo())

Map2 = geemap.Map(center=[23.099, 53.915], zoom=15)
Map2.add_basemap('HYBRID')
Map2.addLayer(slope.clip(study_area),
              {'min': 0, 'max': 15, 'palette': ['green', 'yellow', 'red']}, 'Slope (deg)')
Map2.addLayer(ee.FeatureCollection(plot).style(color='cyan', fillColor='00000000', width=3),
              {}, 'Project plot')
Map2

## 9. Export final maps to Google Drive
Check progress at https://code.earthengine.google.com/tasks

In [ ]:
task1 = ee.batch.Export.image.toDrive(
    image=B['cls'].visualize(min=1, max=5, palette=PAL),
    description='South_Liwa_relative_salinity_map',
    region=study_area, scale=10, maxPixels=1e9)
task1.start()

task2 = ee.batch.Export.image.toDrive(
    image=annual_ndvi.clip(study_area).visualize(
        min=0, max=0.5, palette=['red', 'yellow', 'green']),
    description='South_Liwa_NDVI_2026',
    region=study_area, scale=10, maxPixels=1e9)
task2.start()
print('Export tasks started.')